# W261 EDA - Sunil Thakur

In [0]:
from pyspark.sql.functions import col
print("Welcome to the Class-3 Team-1 W261 final project!") 


# Source Data Files

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

In [0]:
dbutils.fs.rm("dbfs:/tmp/join-data", recurse=True)

In [0]:
display(dbutils.fs.ls("/tmp"))

In [0]:
display(dbutils.fs.ls("dbfs:/mnt/mids-w261/Data"))

In [0]:
# Airline Data    
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/")
display(df_flights)

In [0]:
# Weather data
df_weather = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")
display(df_weather)

In [0]:
print(df_weather.count())

In [0]:
# Stations data      
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
display(df_stations)

In [0]:
# OTPW
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv").cache()
display(df_otpw.limit(10))


# EDA Steps

In [0]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## 1. Sanity Checks

In [0]:
# Count the number of rows
df_otpw.count()

In [0]:
# Count the number of columns
len(df_otpw.columns)

In [0]:
# Summary statistics
summary_stats = dbutils.data.summarize(df_otpw)
display(summary_stats)

In [0]:
# Check duplicate flights
df_otpw = df_otpw.withColumn("flight_id", F.concat_ws('_', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'CRS_DEP_TIME'))
display(df_otpw.groupBy("flight_id").count().filter(F.col("count") > 1))

## 2. Validate Data Type

In [0]:
# Detect numerical columns

df_sample = df_otpw.replace(["", "NA", "NULL"], None)

numerical_expr = "^-+?[0-9]*\\.?[0-9]+([eE][-+]?[0-9]+)?$"

double_expr = "^[-+]?[0-9]*\\.?[0-9]+([eE][-+]?[0-9]+)?$"
int_expr = "^[-+]?[0-9]+$"
date_expr = "^[0-9]{4}-[0-9]{2}-[0-9]{2}$"

exprs = []

for c in df_otpw.columns:

    exprs.append(
        F.mean(F.col(c).rlike(int_expr).cast("int")).alias(c+"_int")
    )

    exprs.append(
        F.mean(F.col(c).rlike(double_expr).cast("int")).alias(c+"_double")
    )

    exprs.append(
        F.mean(F.col(c).rlike(date_expr).cast("int")).alias(c+"_date")
    )

ratios = df_otpw.select(exprs).collect()[0].asDict()

In [0]:
threshold = 0.95
results = []

for col_name in df_otpw.columns:

    int_ratio = ratios[col_name+"_int"]
    double_ratio = ratios[col_name+"_double"]
    date_ratio = ratios[col_name+"_date"]

    if not (int_ratio or double_ratio or date_ratio):
        dtype = "string"
    elif int_ratio >= threshold:
        dtype = "integer"
    elif double_ratio >= threshold:
        dtype = "double"
    elif date_ratio >= threshold:
        dtype = "date"
    else:
        dtype = "string"

    results.append((col_name, dtype, int_ratio, double_ratio, date_ratio))

datatype_df = spark.createDataFrame(
        results,
        ["column","detected_type","int_ratio","double_ratio", "date_ratio"]
    )
display(datatype_df)

In [0]:
df_otpw_clean = df_otpw

In [0]:
def try_cast(df, col_name, target_type, default_value=None):
    """
    Cast column safely. If cast fails, replace with default_value.
    
    Parameters
    ----------
    df : pyspark.sql.DataFrame
    col_name : str
    target_type : str  ('int','double','timestamp','date','string')
    default_value : value used if cast fails
    """
    if target_type == 'int':
        casted_col = F.when(F.col(col_name).rlike(int_expr), F.col(col_name).cast("decimal(38,0)")).otherwise(F.lit(default_value).cast("int")) 
        casted_col = F.when(casted_col.isNotNull(), F.when(casted_col.between(-2147483648, 2147483647), casted_col.cast("int")).otherwise(casted_col.cast('long'))).otherwise(F.lit(default_value).cast("int"))
    elif target_type == 'double':
        casted_col = F.when(F.col(col_name).rlike(double_expr), F.col(col_name).cast("decimal(38,0)")).otherwise(F.lit(default_value).cast("double")) 
        casted_col = F.when(casted_col.isNotNull(), F.when(casted_col.between(-9223372036854775808, 9223372036854775807), casted_col.cast("double")).otherwise(casted_col.cast('decimal(38,0)'))).otherwise(F.lit(default_value).cast("double"))
    else:
        casted_col = F.col(col_name).cast(target_type)

    return df.withColumn(
        col_name,
        F.when(casted_col.isNotNull(), casted_col)
         .otherwise(F.lit(default_value).cast(target_type))
    )

In [0]:
# convert numeric columns appropriately

double_columns = [col_name for (col_name, dtype, int_ratio, double_ratio, date_ratio) in results if dtype == "double"]
int_columns = [col_name for (col_name, dtype, int_ratio, double_ratio, date_ratio) in results if dtype == "integer"]

# convert int columns
for column in int_columns:
    df_otpw_clean = try_cast(df_otpw_clean, column, "int", 0)

# convert double columns
for column in double_columns:
    df_otpw_clean = try_cast(df_otpw_clean, column, "double", 0.0)

display(df_otpw_clean.limit(10))

In [0]:
# Convert date columns
date_cols = [
    "FL_DATE"
]
for c in date_cols:
    df_otpw = df_otpw.withColumn(c, F.to_date(F.col(c)))
    
display(df_otpw.select('FL_DATE').limit(10))

## 3. Target Variable Analysis

In [0]:
df_otpw.select("DEP_DEL15").distinct().show(20, False)

In [0]:

# Convert target variable to integer
df_otpw = df_otpw.withColumn(
    "DEP_DEL15",
    F.col("DEP_DEL15").cast("double").cast("int")
)

# Class distribution
df_otpw.groupBy("DEP_DEL15").count().show()
# df_otpw = df_otpw.withColumn("DEP_DEL15", col("DEP_DEL15").cast("int"))

In [0]:
# Percentage class distribution
total = df_otpw.count()

df_otpw.groupBy("DEP_DEL15").agg(
    (F.count("*")/total).alias("percentage")
).show()

In [0]:
# Severity distribution
display(df_otpw.select("DEP_DELAY"))

In [0]:
target_dist = (
    df_otpw_clean
    .groupBy("DEP_DEL15")
    .count()
    .orderBy("DEP_DEL15")
)

target_pd = target_dist.toPandas()

In [0]:
target_pd["DEP_DEL15"] = target_pd["DEP_DEL15"].fillna("Missing")
target_pd["percent"] = (
    target_pd["count"] / target_pd["count"].sum() * 100
)

In [0]:
# Plot target variable distribution
plt.figure(figsize=(8,6))

ax = sns.barplot(
    x="DEP_DEL15",
    y="count",
    data=target_pd
)

# Add percentage labels
for i,row in target_pd.iterrows():
    ax.text(
        i,
        row["count"] + 10000,
        f'{row["percent"]:.1f}%',
        ha='center',
        fontsize=11
    )

plt.title("Distribution of Flight Departure Delays (DEP_DEL15)")
plt.xlabel("Departure Delay Indicator")
plt.ylabel("Number of Flights")

plt.show()

In [0]:
# Severity histogram
df_otpw_clean = df_otpw_clean.withColumn(
    "DEP_DELAY",
    F.col("DEP_DELAY").cast("double").cast("int")
)

df_otpw.select("DEP_DELAY").describe().show()

## 4. Missing Data Analysis

In [0]:
# Missing values per column
missing_counts = df_otpw.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df_otpw.columns
])

display(missing_counts)

In [0]:
# Missing percentage
total = df_otpw.count()

missing_percent = df_otpw.select([
    (F.count(F.when(col(c).isNull(), c)) / total).alias(c)
    for c in df_otpw.columns
])

display(missing_percent)

In [0]:
total_rows = df_otpw_clean.count()

missing_df = df_otpw_clean.select([
    (F.sum(F.col(c).isNull().cast("int")) / total_rows * 100).alias(c)
    for c in df_otpw.columns
])

missing_percent_df = missing_df.select(
    F.explode(
        F.array([
            F.struct(F.lit(c).alias("column"),
                     F.col(c).alias("missing_percent"))
            for c in missing_df.columns
        ])
    ).alias("col")
).select("col.*")

missing_percent_df = missing_percent_df.filter(F.col("missing_percent") > 0.0)

missing_percent_df.orderBy(F.desc("missing_percent")).show(216, False)

In [0]:
missing_pd = missing_percent_df.toPandas()

missing_pd = missing_pd.sort_values("missing_percent", ascending=False)

plt.figure(figsize=(10,8))
plt.barh(missing_pd["column"][:30], missing_pd["missing_percent"][:30])
plt.xlabel("Missing %")
plt.title("Top 30 Columns by Missing Values")
plt.gca().invert_yaxis()
plt.show()

In [0]:
bins = [-0.01,0,1,5,10,25,50,75,99,100]

labels = [
    "0%",
    "0-1%",
    "1-5%",
    "5-10%",
    "10-25%",
    "25-50%",
    "50-75%",
    "75-99%",
    "99-100%"
]

missing_pd["missing_bin"] = pd.cut(
    missing_pd["missing_percent"],
    bins=bins,
    labels=labels
)

missing_bin_counts = (
    missing_pd
    .groupby("missing_bin")
    .size()
    .reset_index(name="num_columns")
)

plt.figure(figsize=(10,6))

sns.barplot(
    x="missing_bin",
    y="num_columns",
    data=missing_bin_counts,
    color="steelblue"
)

plt.xlabel("Missing Value Percentage Range")
plt.ylabel("Number of Columns")
plt.title("Distribution of Missing Values Across Features")

plt.xticks(rotation=45)

plt.show()

In [0]:
sample_pd = df_otpw_clean.sample(0.01).toPandas()
sns.heatmap(sample_pd.isnull(), cbar=False)

In [0]:
# Remove columns with >90% missing values
missing_columns = missing_percent_df.filter(F.col("missing_percent") >= 90.0).select("column").collect()

missing_columns = [c["column"] for c in missing_columns]
print(missing_columns)

# Drop missing columns
df_otpw_clean = df_otpw_clean.drop(*missing_columns)

# Outliers detection

In [0]:
numeric_cols = []
numeric_cols.extend(int_columns)
numeric_cols.extend(double_columns)
numeric_cols = [field.name for field in df_otpw_clean.schema.fields if field.name in numeric_cols]

In [0]:
outlier_stats = []

for col in numeric_cols:

    quantiles = df_otpw_clean.approxQuantile(col, [0.25, 0.75], 0.01)
    
    if len(quantiles) < 2:
        continue

    q1, q3 = quantiles
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df_otpw_clean.filter(
        (F.col(col) < lower) | (F.col(col) > upper)
    ).count()

    total = df_otpw_clean.count()

    outlier_stats.append((col, outliers/total))

In [0]:
outlier_df = pd.DataFrame(outlier_stats, columns=["feature","outlier_percent"])
outlier_df = outlier_df.sort_values("outlier_percent", ascending=False)

In [0]:
top_outliers = outlier_df.head(20)

plt.figure(figsize=(10,8))

ax = sns.barplot(
    data=top_outliers,
    y="feature",
    x="outlier_percent"
)

plt.xlabel("Outlier Percentage")
plt.ylabel("Feature")
plt.title("Top Features by Outlier Percentage")

for i, v in enumerate(top_outliers["outlier_percent"]):
    ax.text(
        # v + 0.0001,      # slight offset so text appears outside bar
        v/2,
        i,
        f"{v*100:.2f}%",
        ha='center',
        va='center',
        color='white'
    )

plt.show()

## 5. Data Leakage Detection

In [0]:
# Identify obvious leakage columns
leakage_cols = [
    # delay leakage
    "DEP_DELAY",
    "DEP_DELAY_NEW",
    "DEP_DELAY_GROUP",
    "ARR_DELAY",
    "ARR_DELAY_NEW",
    "ARR_DELAY_GROUP",
    "ARR_DEL15",
    # events after departure
    "TAXI_OUT",
    "TAXI_IN",
    "WHEELS_OFF",
    "WHEELS_ON",
    "AIR_TIME",
    "ACTUAL_ELAPSED_TIME",
    # delay cause
    "SECURITY_DELAY",
    "CARRIER_DELAY",
    "NAS_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
    # arrival info
    "ARR_TIME",
    "ARR_TIME_BLK",   
    # aircraft operations
    "FIRST_DEP_TIME",
    "TOTAL_ADD_GTIME",
    "LONGEST_ADD_GTIME",
    # time leakage
    "sched_depart_date_time",
    "sched_depart_date_time_UTC",
    "four_hours_prior_depart_UTC",
    "two_hours_prior_depart_UTC"
]

leakage_cols = [c for c in leakage_cols if c in df_otpw.columns]

df_otpw_clean = df_otpw_clean.drop(*leakage_cols)

## 6. Temporal Pattern Analysis

In [0]:
# Extract departure hour
df_otpw_clean = df_otpw_clean.withColumn(
    "dep_hour",
    F.substring(F.lpad(F.col("CRS_DEP_TIME").cast("string"), 4, "0"), 1, 2).cast('int')
)

In [0]:
# Analyze delay rate vs hour
delay_by_hour = df_otpw_clean.groupBy("dep_hour").agg(
    F.avg("DEP_DEL15").alias("delay_rate")
).orderBy("dep_hour")

display(delay_by_hour)

In [0]:
# Analyze delay rate vs dow
delay_by_dow = df_otpw_clean.groupBy("DAY_OF_WEEK").agg(
    F.avg("DEP_DEL15").alias("delay_rate")
)

display(delay_by_dow)

In [0]:
df_otpw_clean.select("MONTH").distinct().show()

In [0]:
# Analyze delay rate vs month
delay_by_month = df_otpw_clean.groupBy("MONTH").agg(
    F.avg("DEP_DEL15").alias("delay_rate")
)

display(delay_by_month)

In [0]:
hour_data = delay_by_hour.toPandas()
dow_data = delay_by_dow.toPandas()
month_data = delay_by_month.toPandas()

fig, axes = plt.subplots(1,3, figsize=(18,5))

# Delay by hour
sns.lineplot(
    x="dep_hour",
    y="delay_rate",
    data=hour_data,
    marker="o",
    ax=axes[0]
)

axes[0].set_title("Delay Rate by Departure Hour")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Delay Rate")

# Delay by day of week
sns.barplot(
    x="DAY_OF_WEEK",
    y="delay_rate",
    data=dow_data.sort_values("DAY_OF_WEEK"),
    ax=axes[1]
)

axes[1].set_title("Delay Rate by Day of Week")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Delay Rate")

# Delay by month
sns.barplot(
    x="MONTH",
    y="delay_rate",
    data=month_data.sort_values("MONTH"),
    ax=axes[2]
)

axes[2].set_title("Delay Rate by Month")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Delay Rate")

plt.tight_layout()
plt.show()

In [0]:
hour_volume = (
    df_otpw_clean
    .groupBy("dep_hour")
    .count()
    .orderBy("dep_hour")
    .toPandas()
)

dow_volume = (
    df_otpw_clean
    .groupBy("DAY_OF_WEEK")
    .count()
    .orderBy("DAY_OF_WEEK")
    .toPandas()
)

month_volume = (
    df_otpw_clean
    .groupBy("MONTH")
    .count()
    .orderBy("MONTH")
    .toPandas()
)

In [0]:
fig, axes = plt.subplots(1,3, figsize=(18,5))

# Flights by hour
sns.lineplot(
    x="dep_hour",
    y="count",
    data=hour_volume,
    marker="o",
    ax=axes[0]
)

axes[0].set_title("Flight Volume by Departure Hour")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Number of Flights")

# Flights by day of week
sns.barplot(
    x="DAY_OF_WEEK",
    y="count",
    data=dow_volume,
    ax=axes[1]
)

axes[1].set_title("Flight Volume by Day of Week")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Number of Flights")

# Flights by month
sns.barplot(
    x="MONTH",
    y="count",
    data=month_volume,
    ax=axes[2]
)

axes[2].set_title("Flight Volume by Month")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Number of Flights")

plt.tight_layout()
plt.show()

## 7. Airport-Level Analysis

In [0]:
# Flight volume by airport
top_airports = df_otpw_clean.groupBy("ORIGIN") \
    .count() \
    .orderBy("count", ascending=False)

display(top_airports.limit(20))

In [0]:
# Delay rate by airport
delay_airport = df_otpw_clean.groupBy("ORIGIN").agg(
    F.avg("DEP_DEL15").alias("delay_rate"),
    F.count("*").alias("flights")
).filter("flights > 1000")

display(delay_airport.orderBy("delay_rate", ascending=False))

## 8. Airline-Level Analysis

In [0]:
# Delay rate by airline
delay_airline = df_otpw_clean.groupBy("OP_UNIQUE_CARRIER").agg(
    F.avg("DEP_DEL15").alias("delay_rate"),
    F.count("*").alias("flights")
)

display(delay_airline.orderBy("delay_rate", ascending=False))

## 9. Route-Level Analysis

In [0]:
# Create route column
df_otpw_clean = df_otpw_clean.withColumn(
    "route",
    F.concat_ws("-", "ORIGIN", "DEST")
)

In [0]:
# Delay rate by route
delay_route = df_otpw_clean.groupBy("route").agg(
    F.avg("DEP_DEL15").alias("delay_rate"),
    F.count("*").alias("flights")
).filter("flights > 500")

display(delay_route.orderBy("delay_rate", ascending=False))

In [0]:
airline_pd = delay_airline.toPandas()
airport_pd = delay_airport.toPandas()
route_pd = delay_route.orderBy("delay_rate", ascending=False).limit(100).toPandas()



In [0]:
fig, axes = plt.subplots(1,2, figsize=(18,5))

# plt.figure(figsize=(10,7))

sns.scatterplot(
    data=airport_pd,
    x="flights",
    y="delay_rate",
    size="flights",
    sizes=(50,1000),
    alpha=0.7,
    ax=axes[0]
)

axes[0].set_title("Airport Congestion Analysis")
axes[0].set_xlabel("Flight Volume")
axes[0].set_ylabel("Delay Rate")

sns.scatterplot(
    data=route_pd,
    x="flights",
    y="delay_rate",
    size="flights",
    sizes=(40,800),
    alpha=0.7,
    ax=axes[1]
)

axes[1].set_xlabel("Flight Volume")
axes[1].set_ylabel("Delay Rate")
axes[1].set_title("Route Performance: Traffic vs Delay")

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(10,7))

sns.scatterplot(
    data=airline_pd,
    x="flights",
    y="delay_rate",
    size="flights",
    sizes=(50,800),
    alpha=0.7
)

for i,row in airline_pd.iterrows():
    plt.text(row["flights"], row["delay_rate"], row["OP_UNIQUE_CARRIER"])

plt.xlabel("Flight Volume")
plt.ylabel("Delay Rate")
plt.title("Airline Performance: Flight Volume vs Delay Rate")

plt.show()

## 10. Weather Feature Exploration

In [0]:
# Explore weather columns
weather_cols = [
    "HourlyWindSpeed",
    "HourlyVisibility",
    # "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyAltimeterSetting",
    "HourlyStationPressure"
]

# Overlayed boxplots for df_weather
# weather_cols = ['HourlyPrecipitationDouble', 'HourlyVisibilityDouble', 'HourlyWindSpeedDouble']
weather_data = df_otpw_clean.select(*weather_cols).toPandas()

plt.figure(figsize=(10, 6))
weather_data.boxplot(column=weather_cols)
plt.title('Boxplots of Weather Variables')
plt.xlabel('Weather Variables')
plt.ylabel('Values')
plt.xticks(rotation=45)
plt.show()

In [0]:
# Display weather distribution
df_otpw_clean.select(*weather_cols).describe().show()

In [0]:
# Delay rate by wind speed
df_otpw_clean = df_otpw_clean.withColumn(
    "wind_bin",
    F.floor(F.col("HourlyWindSpeed")/5)*5
)

wind_delay = df_otpw_clean.groupBy("wind_bin").agg(
    F.avg("DEP_DEL15").alias("delay_rate"),
    F.count("*").alias("flights")
)

display(wind_delay.orderBy("wind_bin"))

In [0]:
df_weather_sample_pd = (
    df_otpw_clean
    .select(["DEP_DEL15"] + weather_cols)
    .sample(fraction=0.05, seed=42)
    .toPandas()
)

weather_long = pd.melt(
    df_weather_sample_pd,
    id_vars="DEP_DEL15",
    value_vars=weather_cols,
    var_name="Weather_Feature",
    value_name="Value"
)

plt.figure(figsize=(14,8))

sns.boxplot(
    x="Weather_Feature",
    y="Value",
    hue="DEP_DEL15",
    data=weather_long
)

plt.xticks(rotation=45)
plt.title("Weather Variables by Delay Status")
plt.xlabel("Weather Feature")
plt.ylabel("Value")

plt.show()

## 11. Correlation Analysis

In [0]:
df_otpw_clean = df_otpw_clean.withColumn(
    "DISTANCE",
    F.col("DISTANCE").cast("double").cast("int")
)

df_otpw_clean.select("DISTANCE").describe().show()

In [0]:
df_otpw_clean = df_otpw_clean.withColumn(
    "CRS_ELAPSED_TIME",
    F.col("CRS_ELAPSED_TIME").cast("double").cast("int")
)

df_otpw_clean.select("CRS_ELAPSED_TIME").describe().show()

In [0]:
df_otpw_clean.select("DEP_DEL15").describe().show()

In [0]:
numeric_cols = []
numeric_cols.extend(int_columns)
numeric_cols.extend(double_columns)
subset_schema = {field.name: field.dataType for field in df_otpw_clean.schema.fields if field.name in numeric_cols}
print("Subset schema (from schema):", subset_schema)

In [0]:
# Analyze correlation between weather and delay

for c in [col for col in numeric_cols if col in df_otpw_clean.columns]:
    print(c, df_otpw_clean.stat.corr(c, "DEP_DEL15"))

## 12. Explore High Cardinality Features

In [0]:
# Remove obvious ID columns
id_cols = [col for col in df_otpw_clean.columns if col.endswith("_ID") or col.endswith("_WAC")]
print(id_cols)

df_otpw_clean = df_otpw_clean.drop(*id_cols)

# Remove columns with only one value
single_value_cols = []
for c in df_otpw_clean.columns:
    if df_otpw_clean.select(c).distinct().count() == 1:
        single_value_cols.append(c)

single_value_cols = [s for s in single_value_cols if s not in ['QUARTER', 'YEAR']]

df_otpw_clean = df_otpw_clean.drop(*single_value_cols)
  

In [0]:
# Analyze categorical columns for high cardinality

total_rows = df_otpw_clean.count()

cardinality_df = df_otpw_clean.select([
    (F.approx_count_distinct(c) / F.lit(total_rows)).alias(c)
    for c in df_otpw_clean.columns
])

cardinality_df = cardinality_df.select(
    F.explode(
        F.array([
            F.struct(F.lit(c).alias("column"), F.col(c).alias("cardinality_ratio"))
            for c in cardinality_df.columns
        ])
    )
).select("col.*")

cardinality_df.orderBy(F.desc("cardinality_ratio")).display()

# Avoid one-hot encoding for high cardinality features.

In [0]:
cardinality_pd = cardinality_df.toPandas()
plt.hist(cardinality_pd["cardinality_ratio"], bins=30)
plt.xlabel("Cardinality Ratio")
plt.ylabel("Number of Columns")
plt.title("Distribution of Column Cardinality")

## 13. Dataset Balance Across Years

In [0]:
flights_by_year = df_otpw_clean.groupBy("YEAR").count()

display(flights_by_year)

## 14. Flight Distance Impact

In [0]:
df_otpw_clean = df_otpw_clean.withColumn(
    "distance_bin",
    F.floor(F.col("DISTANCE")/500)*500
)

distance_delay = df_otpw_clean.groupBy("distance_bin").agg(
    F.avg("DEP_DEL15").alias("delay_rate"),
    F.count("*").alias("flights")
)

display(distance_delay.orderBy("distance_bin"))

## 15. Cancellation Analysis

In [0]:
df_otpw_clean = df_otpw_clean.withColumn(
    "CANCELLED",
    F.col("CANCELLED").cast("double").cast("int")
)

df_otpw_clean.select("CANCELLED").describe().show()

In [0]:
df_otpw_clean.groupBy("CANCELLED").count().show()
df_otpw_clean.groupBy("CANCELLED").agg(
    F.avg("DEP_DEL15").alias("delay_rate")
).show()

## 16. Predictive Signal Tests

In [0]:
display(
    df_otpw_clean.groupBy("dep_hour")
    .agg(F.avg("DEP_DEL15").alias("delay_rate"))
)

In [0]:
precip_delay = df_otpw_clean.groupBy("HourlyPrecipitation").agg(
    F.avg("DEP_DEL15").alias("delay_rate")
)

display(precip_delay)

## 17. Other Exploratory Analysis

In [0]:
# delay rate vs hour

# delay rate vs airline

# delay rate vs airport

# delay vs month

# wind speed vs delay

# precipitation vs delay

# distance vs delay

# flight volume by airport

df_sample_pd = df_otpw_clean.sample(0.01, seed=42).toPandas()

In [0]:
sns.lineplot(
    data=df_sample_pd.groupby("dep_hour")["DEP_DEL15"].mean().reset_index(),
    x="dep_hour",
    y="DEP_DEL15"
)

In [0]:
sns.barplot(x="DAY_OF_WEEK", y="DEP_DEL15", data=df_sample_pd)

In [0]:
sns.lineplot(
    data=df_sample_pd.groupby("MONTH")["DEP_DEL15"].mean().reset_index(),
    x="MONTH",
    y="DEP_DEL15"
)

In [0]:
sns.histplot(df_sample_pd["distance_bin"], bins=50, kde=True)

In [0]:
sns.lineplot(
    data=df_sample_pd.groupby("distance_bin")["DEP_DEL15"].mean().reset_index(),
    x="distance_bin",
    y="DEP_DEL15"
)

In [0]:
weather_cols = [
    "HourlyWindSpeed",
    "HourlyVisibility",
    # "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyAltimeterSetting",
    "HourlyStationPressure"
]

fig, axes = plt.subplots(3,3, figsize=(15,10))

for i, col in enumerate(weather_cols):
    sns.boxplot(
        x="DEP_DEL15",
        y=col,
        data=df_sample_pd,
        ax=axes[i//3, i%3]
    )
    axes[i//3, i%3].set_title(col)

plt.tight_layout()

In [0]:
daily_delay = df_sample_pd.groupby("FL_DATE")["DEP_DEL15"].mean()

daily_delay.plot(figsize=(12,6))

In [0]:
df_otpw_clean.columns

In [0]:
numeric_cols = []
numeric_cols.extend(int_columns)
numeric_cols.extend(double_columns)
numeric_cols = [field.name for field in df_otpw_clean.schema.fields if field.name in numeric_cols]
numeric_cols.extend(['dep_hour', 'wind_bin', 'distance_bin'])
print(numeric_cols)


In [0]:
corr_cols = list(set([col for col in numeric_cols if col in df_otpw_clean.columns]))
corr_cols = [col for col in corr_cols if col not in ["DEP_TIME", "_row_desc"]]
corr = df_sample_pd[corr_cols].corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap="coolwarm", center=0)

# corr_target = corr["DEP_DEL15"].sort_values(ascending=False)
# corr_target.plot.bar(figsize=(10,6))

In [0]:
corr_target = corr["DEP_DEL15"].sort_values(ascending=False)
corr_target.plot.bar(figsize=(10,6))

In [0]:
top_corr = corr["DEP_DEL15"].abs().sort_values(ascending=False).head(20)
sns.barplot(x=top_corr.values, y=top_corr.index)

## Checkpoint cleaned OTPW data

In [0]:
# Create folder
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

# Save df_weather as a parquet file
df_otpw_clean.write.mode('overwrite').parquet(f"{folder_path}/otpw_clean.parquet")